##### Import the required modules and configure the system path to locate them

In [ ]:
import sys
sys.path.append("../../utils")
import networkx
import yaml
import os
import subprocess
import pandas as pd
import astra_sim_sdk.astra_sim_sdk as astra_sim_kit
from common import FileFolderUtils
from IPython.display import IFrame
from astra_sim import AstraSim, Collective, NetworkBackend
from infragraph.blueprints.devices.nvidia.dgx import NvidiaDGX
from infragraph.blueprints.fabrics.single_tier_fabric import SingleTierFabric
from infragraph.infragraph_service import InfraGraphService

##### Call the AstraSim client helper with the server endpoint and tag to connect to the ASTRA-sim gRPC server, initialize the SDK, and create a tagged folder for configs, results, and logs

In [ ]:
astra = AstraSim(server_endpoint = "172.17.0.2:8989", tag = "ns3_single_tier_with_dgx")

##### Create a single tier rack device with two Nvidia DGX and a single switch using infragraph device, fabric blueprint

In [ ]:
dgx_count = 2
server = NvidiaDGX()
infrastructure = SingleTierFabric(server, dgx_count)
astra.configuration.infragraph.infrastructure.deserialize(infrastructure.serialize())

##### Initialize the Infragraph service, display the fabric topology, and retrieve/set the total number of NPUs to generate the collective

In [ ]:
service = InfraGraphService()
service.set_graph(infrastructure)

g = service.get_networkx_graph()
print(networkx.write_network_text(g, vertical_chains=True))

total_npus = service.get_component(server, "xpu").count * dgx_count
print(total_npus)

##### Save infragraph as a yaml

In [ ]:
with open(os.path.join(FileFolderUtils.get_instance().OUTPUT_DIR,"../infrastructure",f"{astra.tag}.yaml"),"w") as f:
    data = infrastructure.serialize("dict")
    yaml.dump(data, f, default_flow_style=False, indent=4)

print("saved yaml to:", os.path.join(FileFolderUtils.get_instance().OUTPUT_DIR,"..",f"{astra.tag}.yaml"))

##### Visualize the Infragraph

In [ ]:
# VISUALIZER_START
PORT = 8765

subprocess.run(
    f"lsof -ti:{PORT} | xargs -r kill -9",
    shell=True
)

infra_yaml_path = os.path.join(
    FileFolderUtils.get_instance().OUTPUT_DIR,
    "../infrastructure",
    f"{astra.tag}.yaml"
)

infra_dir = os.path.dirname(infra_yaml_path)
visual_output_dir = os.path.normpath(os.path.join(infra_dir, "../visuals"))

subprocess.run(
    ["infragraph", "visualize", "--input", infra_yaml_path, "--output", visual_output_dir],
    check=True
)

subprocess.Popen(
    ["python3", "-m", "http.server", f"{PORT}"],
    cwd=visual_output_dir
)
IFrame(f"http://localhost:{PORT}/index.html", width="100%", height=700)
# VISUALIZER_END

##### Generate workload execution traces for each rank and set the required data size for AstraSim configuration

In [ ]:
astra.configuration.common_config.workload = astra.generate_collective(collective=Collective.ALLREDUCE, coll_size= 8 *1024*1024, npu_range=[0, total_npus])

##### Configure ASTRA-sim system config

In [ ]:
astra.configuration.common_config.system.scheduling_policy = astra.configuration.common_config.system.LIFO
astra.configuration.common_config.system.endpoint_delay = 10
astra.configuration.common_config.system.active_chunks_per_dimension = 1
astra.configuration.common_config.system.all_gather_implementation = [astra.configuration.common_config.system.RING]
astra.configuration.common_config.system.all_to_all_implementation = [astra.configuration.common_config.system.DIRECT]
astra.configuration.common_config.system.all_reduce_implementation = [astra.configuration.common_config.system.ONERING]
astra.configuration.common_config.system.collective_optimization = astra.configuration.common_config.system.LOCALBWAWARE
astra.configuration.common_config.system.local_mem_bw = 1600

##### Configure ASTRA-sim remote memory configuration

In [ ]:
astra.configuration.common_config.remote_memory.memory_type = astra.configuration.common_config.remote_memory.NO_MEMORY_EXPANSION
print(astra.configuration.common_config.remote_memory)

##### Configure the selected network backend and the topology (infragraph or nc_topology)

In [ ]:
astra.configuration.network_backend.choice = astra.configuration.network_backend.NS3
astra.configuration.network_backend.ns3.topology.choice = astra.configuration.network_backend.ns3.topology.INFRAGRAPH
astra.configuration.network_backend.ns3.network.packet_payload_size = int(8192)

##### Adding ns3 trace and logical dimension 

In [ ]:
astra.configuration.network_backend.ns3.logical_topology.logical_dimensions = [total_npus]
astra.configuration.network_backend.ns3.trace.trace_ids = []
for i in range(0, total_npus):
    astra.configuration.network_backend.ns3.trace.trace_ids.append(i)

##### Adding ASTRA-sim - Infragraph specific annotation

In [ ]:
host_device_spec = astra_sim_kit.AnnotationDeviceSpecifications()
host_device_spec.device_bandwidth_gbps = 100
host_device_spec.device_latency_ms = 0.05
host_device_spec.device_name = server.name
host_device_spec.device_type = "host"
astra.configuration.infragraph.annotations.device_specifications.append(host_device_spec)

switch_device_spec = astra_sim_kit.AnnotationDeviceSpecifications()
switch_device_spec.device_bandwidth_gbps = 100
switch_device_spec.device_latency_ms = 0.05
switch_device_spec.device_name = "switch"
switch_device_spec.device_type = "switch"
astra.configuration.infragraph.annotations.device_specifications.append(
    switch_device_spec
)

##### Configure ASTRA-sim cmd parameters

In [ ]:
astra.configuration.common_config.cmd_parameters.comm_scale = 1
astra.configuration.common_config.cmd_parameters.injection_scale = 1
astra.configuration.common_config.cmd_parameters.rendezvous_protocol = False

#### Start the simulation by specifying the network backend

In [ ]:
astra.run_simulation(NetworkBackend.NS3)

##### Download all the configurations as a zip

In [ ]:
astra.download_configuration()

##### Read output files

In [ ]:
df = pd.read_csv(os.path.join(FileFolderUtils.get_instance().OUTPUT_DIR, "fct.csv"))
df.head()